# SMARTBind benchmark on 5-fold cross-validation HARIBOSS dataset

HARIBOSS 5-fold cross validation dataset can be found: https://drive.google.com/drive/folders/1j7QMGHKyhpJLasyXLjYZ2Db9SbvKNKbR?usp=sharing <br>
Their corresonding benchmarked SMARTBind weights can be found: https://drive.google.com/drive/folders/1AOx9_sbvRHL4GKpQ3MCOZOcD-d3jw_iI?usp=sharing

GerNA-Bind subset of HARIBOSS 5-fold cross-validation dataset can be found: https://drive.google.com/drive/folders/1V5jDPI4scKlNuW9RbB7uS90xDNT15Oke?usp=sharing <br>
Their corresonding benchmarked SMARTBind weights can be found: https://drive.google.com/drive/folders/1XG6ORkcHEpImc_clgVxSVsOTjySsJp9x?usp=sharing

The reproducing pipeline uses pair-based split as an example.


In [7]:
import sys
sys.path.append('../../../..')
from smartbind.preprocess import build_val_test_set
from smartbind.dataloader import RLDataLoader, RLDataset
from smartbind import load_smartbind_models

import tqdm
from smartbind.model.margin import sigmoid_cosine_distance_test
import torch
import numpy as np
from sklearn.metrics import roc_auc_score

In [11]:
current_path = 'hariboss_5fd_by_pair_split_clean.pkl'
weight_dir = 'weights/pairSplit'

topk_decoy = 100
batch_size = 24
num_workers = 2
decoy_num = 48
replace_res_type = ['R', 'Y', 'K', 'M', 'S', 'W', 'B', 'D', 'H', 'V', 'N', '-']
device = 'cpu'

In [9]:
full_rank_percentile_list = []
full_auroc_list = []
for current_fold_num in range(1, 6):
    print(f'\nProcessing fold {current_fold_num}')
    val_rna_seq_list, val_match_smol_list, val_rna_name_list, val_match_smol_name_list, val_binding_index_list, \
        val_match_smol_fp_list, train_rna_seq_list, train_match_smol_list, train_rna_name_list, \
        train_match_smol_name_list, train_binding_index_list, train_match_smol_fp_list = (
                                        build_val_test_set(
                                            fold_num=current_fold_num-1,
                                            dict_path=current_path,
                                            topk_decoy=topk_decoy))
    val_dataset = RLDataset(rna_sequences=val_rna_seq_list,
                                rna_sequences_names=val_rna_name_list,
                                match_smols=val_match_smol_fp_list,
                                match_smols_names=val_match_smol_name_list,
                                non_binding_index_list=val_binding_index_list,
                                is_val=True)
    val_dataloader = RLDataLoader(dataset=val_dataset,
                                    batch_size=batch_size,
                                    num_workers=num_workers,
                                    if_shuffle=False)
    # load model
    weight_path = f'{weight_dir}/fold{current_fold_num}.pth'
    smartbind_model = load_smartbind_models(
        model_path=weight_path,
        device=device,
        vs_mode=True
        )
    # loop validation dataloader and perform inference
    rank_percentile_list = []
    auroc_list = []
    smartbind_model.eval()
    with torch.no_grad():
        for batch in tqdm.tqdm(val_dataloader.dataloader):
            rna_sequences, match_smols, decoy_smols_list, _, _, _ = batch
            
            # Process RNA sequences
            rna_tokenized_list = []
            for rna_sequence in rna_sequences:
                rna_tokenized = smartbind_model.inference_single_rna(rna_sequence[1])
                rna_tokenized_list.append(rna_tokenized)
            
            # Process each RNA-ligand pair
            for anchor_rna, match_smol, decoy_smols in zip(
                rna_tokenized_list, match_smols, decoy_smols_list
            ):
                if len(decoy_smols) == 0:
                    print('No decoy smols for this pair, skip.')
                    continue
                # else:
                #     print(f'Processing pair with {len(decoy_smols)} decoys.')
                
                # Process match smol - use inference_list_smols with single item
                anchor_rna = anchor_rna.squeeze(1)
                match_smol_tokenized = smartbind_model.inference_list_smols([match_smol[1]])[0]
                match_smol_tokenized = match_smol_tokenized.unsqueeze(dim=0)
                
                # Calculate positive distance
                match_distance = sigmoid_cosine_distance_test(anchor_rna, match_smol_tokenized)
    
                decoy_smol_tokenized_list = smartbind_model.inference_list_smols(decoy_smols)
                
                # Calculate distances
                decoy_distance_list = []
                for decoy_smol_tokenized in decoy_smol_tokenized_list:
                    decoy_smol_tokenized = decoy_smol_tokenized.unsqueeze(dim=0)
                    decoy_distance = sigmoid_cosine_distance_test(anchor_rna, decoy_smol_tokenized)
                    decoy_distance_list.append(decoy_distance)
                
                predicted_scores = [match_distance.item()] + [decoy_distance.item() for decoy_distance in decoy_distance_list]
                ground_truth_labels = [1] + [0] * len(decoy_distance_list)
                auroc = roc_auc_score(ground_truth_labels, predicted_scores)
                auroc_list.append(auroc)

                # Calculate rank percentile
                rank = 0
                for decoy_distance in decoy_distance_list:
                    if match_distance.item() <= decoy_distance.item():
                        rank += 1
                rank_percentile = (len(decoy_distance_list) + 1 - rank) / (len(decoy_distance_list) + 1)

                rank_percentile_list.append(rank_percentile)

    print(f'Inference completed! Total pairs: {len(rank_percentile_list)}')
    # Display rank percentile statistics
    mean_rank = np.mean(rank_percentile_list)
    median_rank = np.median(rank_percentile_list)
    print(f'Mean Rank Percentile: {mean_rank:.4f}')
    print(f'Median Rank Percentile: {median_rank:.4f}')
    full_rank_percentile_list.extend(rank_percentile_list)
    # Display AUROC statistics
    if len(auroc_list) > 0:
        mean_auroc = np.mean(auroc_list)
        median_auroc = np.median(auroc_list)
        print(f'Mean AUROC: {mean_auroc:.4f}')
        print(f'Median AUROC: {median_auroc:.4f}')
    else:
        print('No valid AUROC scores calculated for this fold.')
    full_auroc_list.extend(auroc_list)


Processing fold 1


Global seed set to 42


Not loaded: ['smol_binding_featurizer.0.weight', 'smol_binding_featurizer.0.bias', 'smol_binding_featurizer.1.weight', 'smol_binding_featurizer.1.bias', 'smol_binding_featurizer.1.running_mean', 'smol_binding_featurizer.1.running_var', 'smol_binding_featurizer.1.num_batches_tracked', 'binding_site_featurizer.cross_attention.smol_projection.weight', 'binding_site_featurizer.cross_attention.smol_projection.bias', 'binding_site_featurizer.cross_attention.projection.weight', 'binding_site_featurizer.cross_attention.projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.weight', 'binding_site_featurizer.cross_attention.batch_norm_projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_mean', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_var', 'binding_site_featurizer.cross_attention.batch_norm_projection.num_batches_tracked', 'binding_site_featurizer.cross_attention.self_attention.0.0.in_proj_weight', 'bindi

100%|██████████| 7/7 [01:06<00:00,  9.56s/it]
Global seed set to 42


Inference completed! Total pairs: 162
Mean Rank Percentile: 0.6512
Median Rank Percentile: 0.7201
Mean AUROC: 0.6433
Median AUROC: 0.7125

Processing fold 2
Not loaded: ['smol_binding_featurizer.0.weight', 'smol_binding_featurizer.0.bias', 'smol_binding_featurizer.1.weight', 'smol_binding_featurizer.1.bias', 'smol_binding_featurizer.1.running_mean', 'smol_binding_featurizer.1.running_var', 'smol_binding_featurizer.1.num_batches_tracked', 'binding_site_featurizer.cross_attention.smol_projection.weight', 'binding_site_featurizer.cross_attention.smol_projection.bias', 'binding_site_featurizer.cross_attention.projection.weight', 'binding_site_featurizer.cross_attention.projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.weight', 'binding_site_featurizer.cross_attention.batch_norm_projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_mean', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_var', 'binding_si

...

Inference completed! Total pairs: 147
Mean Rank Percentile: 0.7728
Median Rank Percentile: 0.8367
Mean AUROC: 0.7681
Median AUROC: 0.8333

Processing fold 4
Not loaded: ['smol_binding_featurizer.0.weight', 'smol_binding_featurizer.0.bias', 'smol_binding_featurizer.1.weight', 'smol_binding_featurizer.1.bias', 'smol_binding_featurizer.1.running_mean', 'smol_binding_featurizer.1.running_var', 'smol_binding_featurizer.1.num_batches_tracked', 'binding_site_featurizer.cross_attention.smol_projection.weight', 'binding_site_featurizer.cross_attention.smol_projection.bias', 'binding_site_featurizer.cross_attention.projection.weight', 'binding_site_featurizer.cross_attention.projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.weight', 'binding_site_featurizer.cross_attention.batch_norm_projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_mean', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_var', 'binding_si

100%|██████████| 7/7 [00:30<00:00,  4.29s/it]
Global seed set to 42


Inference completed! Total pairs: 147
Mean Rank Percentile: 0.7821
Median Rank Percentile: 0.8308
Mean AUROC: 0.7795
Median AUROC: 0.8438

Processing fold 5
Not loaded: ['smol_binding_featurizer.0.weight', 'smol_binding_featurizer.0.bias', 'smol_binding_featurizer.1.weight', 'smol_binding_featurizer.1.bias', 'smol_binding_featurizer.1.running_mean', 'smol_binding_featurizer.1.running_var', 'smol_binding_featurizer.1.num_batches_tracked', 'binding_site_featurizer.cross_attention.smol_projection.weight', 'binding_site_featurizer.cross_attention.smol_projection.bias', 'binding_site_featurizer.cross_attention.projection.weight', 'binding_site_featurizer.cross_attention.projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.weight', 'binding_site_featurizer.cross_attention.batch_norm_projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_mean', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_var', 'binding_si

100%|██████████| 7/7 [00:29<00:00,  4.25s/it]

Inference completed! Total pairs: 147
Mean Rank Percentile: 0.7284
Median Rank Percentile: 0.8226
Mean AUROC: 0.7241
Median AUROC: 0.8279


In [10]:
# print out overall statistics: Mean, Median, Standard Deviation, 95% Confidence Interval
print(f'\nOverall Mean Rank Percentile: {np.mean(full_rank_percentile_list):.4f}')
print(f'Overall Median Rank Percentile: {np.median(full_rank_percentile_list):.4f}')
print(f'Overall Standard Deviation Rank Percentile: {np.std(full_rank_percentile_list):.4f}')
# Calculate 95% confidence interval for rank percentile
if len(full_rank_percentile_list) > 1:
    mean_rank = np.mean(full_rank_percentile_list)
    std_rank = np.std(full_rank_percentile_list)
    ci_lower = mean_rank - 1.96 * std_rank / np.sqrt(len(full_rank_percentile_list))
    ci_upper = mean_rank + 1.96 * std_rank / np.sqrt(len(full_rank_percentile_list))
    print(f'Overall 95% Confidence Interval Rank Percentile: [{ci_lower:.4f}, {ci_upper:.4f}]')

if len(full_auroc_list) > 0:
    print(f'Overall Mean AUROC: {np.mean(full_auroc_list):.4f}')
    print(f'Overall Median AUROC: {np.median(full_auroc_list):.4f}')
    print(f'Overall Standard Deviation AUROC: {np.std(full_auroc_list):.4f}')
    # Calculate 95% confidence interval for AUROC
    mean_auroc = np.mean(full_auroc_list)
    std_auroc = np.std(full_auroc_list)
    ci_lower = mean_auroc - 1.96 * std_auroc / np.sqrt(len(full_auroc_list))
    ci_upper = mean_auroc + 1.96 * std_auroc / np.sqrt(len(full_auroc_list))
    print(f'Overall 95% Confidence Interval AUROC: [{ci_lower:.4f}, {ci_upper:.4f}]')
else:
    print('No valid AUROC scores calculated across all folds.')


Overall Mean Rank Percentile: 0.7298
Overall Median Rank Percentile: 0.7959
Overall Standard Deviation Rank Percentile: 0.2478
Overall 95% Confidence Interval Rank Percentile: [0.7121, 0.7475]
Overall Mean AUROC: 0.7248
Overall Median AUROC: 0.7917
Overall Standard Deviation AUROC: 0.2528
Overall 95% Confidence Interval AUROC: [0.7067, 0.7428]
